In [41]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import json
import re
from selenium.webdriver.common.keys import Keys

In [42]:
def to_json(data):
    """
    This function takes list of dictionary as Input and 
    then Creates a JSON file in which Input data is stored
    """
    with open("data_dict.json", "w") as outfile:
        json.dump(data, outfile,indent=4)
        outfile.close()

In [43]:
data_list = []
url = "https://www.fdic.gov/bank/individual/failed/banklist.html"
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized") 
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--log-level=3")

In [44]:
def get_data(slug_name):
    driver = webdriver.Chrome(options=options)
    driver.get(url)
    search = driver.find_element(By.XPATH, f'//div[3]/label/select')
    search.send_keys("All")
    search.send_keys(Keys.RETURN)
    time.sleep(3)
    list1 = driver.find_elements(By.XPATH, f'//div[2]/table/tbody/tr')
    for i in range(1, len(list1)+1):
        list2 = driver.find_elements(By.XPATH, f'//div[2]/table/tbody/tr[{i}]/td')
        description = ""
        importantDates = ""
        data_dict ={}
        print("*"*50)
        for j in range(1, len(list2)+1):
            if j==1:
                fullName = driver.find_element(By.XPATH, f'//div[2]/table/tbody/tr[{i}]/td[{j}]').text
                print(fullName)
                referenceUrls = driver.find_element(By.XPATH, f'//div[2]/table/tbody/tr[{i}]/td[{j}]/a').get_attribute("href")
                driver.execute_script("window.open('');")
                driver.switch_to.window(driver.window_handles[1])
                driver.get(referenceUrls)
                time.sleep(2)
                try:
                    description = driver.find_element(By.XPATH, f'/html/body/main/section[1]/div/div/p[2]').text
                    print(description)
                except:
                    try:
                        list3 = driver.find_elements(By.XPATH, f'/html/body/div[4]/p')
                        for k in range (1, len(list3)+1):
                            description = description + " " + driver.find_element(By.XPATH, f'/html/body/div[4]/p[{k}]').text
                        decription = description.replace("\n", "").strip()
                        print(description)
                        try:
                            list4 = driver.find_elements(By.XPATH, f'/html/body/div[4]/div[1]/div[2]/p')
                            for l in range(1, len(list4)+1):
                                text = driver.find_element(By.XPATH, f'/html/body/div[4]/div[1]/div[2]/p[{l}]').text
                                if "Notice of Termination" in text:
                                    EffectiveDate = driver.find_element(By.XPATH, f'/html/body/div[4]/div[1]/div[2]/p[{l+1}]').text
                                if "Notice of Intent to Terminate" in text:
                                    PublicationDate = driver.find_element(By.XPATH, f'/html/body/div[4]/div[1]/div[2]/p[{l+1}]').text
                            importantDates = importantDates + " Notice of Termination: " + EffectiveDate + "; Notice of Intent to Terminate: " + PublicationDate 
                        except:
                            pass
                    except:
                        pass
                driver.close()
                driver.switch_to.window(driver.window_handles[0])
            elif j==2:
                city = driver.find_element(By.XPATH, f'//div[2]/table/tbody/tr[{i}]/td[{j}]').text
            elif j==3:
                state = driver.find_element(By.XPATH, f'//div[2]/table/tbody/tr[{i}]/td[{j}]').text
            elif j==4:
                cert = driver.find_element(By.XPATH, f'//div[2]/table/tbody/tr[{i}]/td[{j}]').text
            elif j==5:
                AccquiringInstitute = driver.find_element(By.XPATH, f'//div[2]/table/tbody/tr[{i}]/td[{j}]').text.replace("No Acquirer", "None")
            elif j==6:
                closingDate = driver.find_element(By.XPATH, f'//div[2]/table/tbody/tr[{i}]/td[{j}]').text
            elif j==7:
                fund = driver.find_element(By.XPATH, f'//div[2]/table/tbody/tr[{i}]/td[{j}]').text

        importantDates = importantDates + "; Closing Date: " + closingDate
        additionalInfo = "Cert: " + cert + "; Acquiring Institution: " + AccquiringInstitute + "; Fund: " + fund
        if "None" not in AccquiringInstitute:
            summary = fullName + " is in the United States - US-U.S FDIC Failed Bank list. " + "The bank was closed on "+ closingDate + " and was acquired by "+ AccquiringInstitute
        else:
            summary = fullName + " is in the United States - US-U.S FDIC Failed Bank list. " + "The bank was closed on "+ closingDate 
        if fullName:
            data_dict['fullName'] = fullName
        if city:
            data_dict['city'] = city
        if state:
            data_dict['state'] = state
        if importantDates:
            data_dict['importantDates'] = importantDates
        if description:
            data_dict['description'] = description
        if additionalInfo:
            data_dict['additionalInfo'] = additionalInfo
        if referenceUrls:
            data_dict['referenceUrls'] = referenceUrls
        if summary:
            data_dict['summary'] = summary
        data_list.append(data_dict)
    return data_list

In [45]:
if __name__ == '__main__':
    data_list = get_data("add_slug_name")
    to_json(data_list)

**************************************************
Almena State Bank
On Friday, October 23, 2020, Almena State Bank was closed by the Kansas Office of the State Bank Commissioner. The FDIC was named Receiver. No advance notice is given to the public when a financial institution is closed. Equity Bank, Andover, KS acquired all deposit accounts and substantially all the assets. All shares of stock were owned by the holding company, which was not involved in this transaction.
**************************************************
First City Bank of Florida
 On Friday, October 16, 2020, First City Bank of Florida was closed by the Florida Office of Financial Regulation. The FDIC was named Receiver. United Fidelity Bank, fsb, Evansville, IN acquired all deposit accounts. The FDIC as Receiver for First City Bank of Florida, Fort Walton Beach, FL has taken all actions necessary to terminate the Receivership Estate. The Receiver published a legal notice of intent to terminate the receivership on S

**************************************************
First NBC Bank
 Please be advised you will not receive any email notification to claim/unlock/unsuspend your account or to provide any private information. Please be aware of any Phishing Scams to obtain information from you. On Friday, April 28, 2017, First NBC Bank, New Orleans, LA was closed by the Louisiana Office of Financial Institutions, and the Federal Deposit Insurance Corporation (FDIC) was named Receiver. No advance notice is given to the public when a financial institution is closed. The FDIC has assembled useful information regarding your relationship with this institution. Besides a checking account, you may have Certificates of Deposit, a car loan, a business checking account, a commercial loan, a Social Security direct deposit, and other relationships with the institution. The FDIC has compiled the following information, which should answer many of your questions. Back to top The FDIC has issued a press release (PR-034-

**************************************************
Proficio Bank
 Please be advised you will not receive any email notification to claim/unlock/unsuspend your account or to provide any private information. Please be aware of any Phishing Scams to obtain information from you. On Friday, March 3, 2017, Proficio Bank, Cottonwood Heights, UT was closed by the Utah Department of Financial Institutions, and the Federal Deposit Insurance Corporation (FDIC) was named Receiver. No advance notice is given to the public when a financial institution is closed. The FDIC has assembled useful information regarding your relationship with this institution. Besides a checking account, you may have Certificates of Deposit, a car loan, a business checking account, a commercial loan, a Social Security direct deposit, and other relationships with the institution. The FDIC has compiled the following information, which should answer many of your questions. Back to top The FDIC has issued a press release (PR-0

**************************************************
Harvest Community Bank
 On Friday, January 13, 2017, Harvest Community Bank was closed by the New Jersey Department of Banking and Insurance. The FDIC was named Receiver. First-Citizens Bank & Trust Company acquired all deposit accounts. The FDIC as Receiver for Harvest Community Bank, Pennsville, NJ has taken all actions necessary to terminate the Receivership Estate. The Receiver published a legal notice of intent to terminate the receivership on February 5, 2019 and has made all dividend distributions required by law. Effective September 1, 2019, the Receiver was discharged and the Receivership Estate was terminated and ceased existence as a legal entity.
**************************************************
Allied Bank
 On Friday, September 23, 2016, Allied Bank was closed by the Arkansas State Bank Department. The FDIC was named Receiver. Arkansas State Bank Department acquired all deposit accounts. The FDIC as Receiver for Allied Ba

**************************************************
Capitol City Bank & Trust Company
 On February 13, 2015, Capitol City Bank & Trust Company was closed by the Georgia Department of Banking & Finance. The FDIC was named Receiver. First-Citizens Bank & Trust Company, Raleigh, NC acquired all deposit accounts. The FDIC as Receiver for Capitol City Bank & Trust Company, Atlanta, GA has taken all actions necessary to terminate the Receivership Estate. The Receiver published a legal notice of intent to terminate the receivership on September 27, 2019 and has made all dividend distributions required by law. Effective February 01, 2020, the Receiver was discharged and the Receivership Estate was terminated and ceased existence as a legal entity.
**************************************************
Highland Community Bank
 On Friday, January 23, 2015, Highland Community Bank, Chicago, IL was closed by the Illinois Department of Financial & Professional Regulation – Division of Banking. The FDIC 

...

**************************************************
Universal Federal Savings Bank
 Please be advised you will not receive any email notification to claim/unlock/unsuspend your account or to provide any private information. Please be aware of any Phishing Scams to obtain information from you. On June 27, 2002, Universal Federal Savings Bank ("Universal FSB") was closed by the Office of Thrift Supervision and the Federal Deposit Insurance Corporation (FDIC) was named Receiver. As Receiver, the FDIC is charged with winding up the business affairs of the failed financial institution. This includes the disposition of assets and liabilities of the failed financial institution and payment of dividends to approved creditors in order of priority. The FDIC, as Receiver, has taken all necessary actions to conclude the affairs of the failed financial institution, made all dividend distributions as required by law and the receivership is deemed terminated. Back to top The FDIC has issued a press re

**************************************************
NextBank, NA
 The FDIC, as receiver for NextBank, National Association (N.A.), has advised the Bank of New York, as Trustee for the NextCard Credit Card Master Trust, that an automatic event, default, acceleration or early amortization based solely on insolvency or appointment of the FDIC as receiver is not enforceable against the FDIC. The FDIC has not indicated any question or doubt to the Trustee or otherwise regarding the validity or effect of any other trigger events for an early amortization of any NextBank Credit Card Trust obligations, including any other such trigger events based on collateral performance. The FDIC reconfirms, under established FDIC policy as codified by express rule at 12 CFR 360.6, that it will not disaffirm or repudiate the completed transfer of financial assets by NextBank, N.A. in securitizations in accord with that FDIC rule. Back to top On February 7, 2002, NextBank, N.A. was closed by Office of the Com

**************************************************
Hamilton Bank, NA
En Español
 En Español Please be advised you will not receive any email notification to claim/unlock/unsuspend your account or to provide any private information. Please be aware of any Phishing Scams to obtain information from you. On January 11, 2002, Hamilton Bank, National Association (N.A.), Miami, FL was closed by Office of the Comptroller of the Currency and the Federal Deposit Insurance Corporation (FDIC) was named Receiver. As Receiver, the FDIC is charged with winding up the business affairs of the failed financial institution. This includes the disposition of assets and liabilities of the failed financial institution and payment of dividends to approved creditors in order of priority. The FDIC, as Receiver, has taken all necessary actions to conclude the affairs of the failed financial institution, made all dividend distributions as required by law and the receivership is deemed terminated. Back to top The 

**************************************************
Sinclair National Bank
 On September 7, 2001, Sinclair National Bank, Gravette, AR was closed by the Office of the Comptroller of the Currency, and the Federal Deposit Insurance Corporation (FDIC) was named Receiver. As Receiver, the FDIC is charged with winding up the business affairs of the failed financial institution. This includes the disposition of assets and liabilities of the failed financial institution and payment of dividends to approved creditors in order of priority. The FDIC, as Receiver, has taken all necessary actions to conclude the affairs of the failed financial institution, made all dividend distributions as required by law and the receivership estate is deemed terminated. Back to top The FDIC issued a press release about the institution’s closure.  If you represent a media outlet and would like information about the closure, please contact the FDIC Public Affairs Office at (202) 898-6993. Back to top All insured de

**************************************************
National State Bank of Metropolis
 On December 14, 2000, The National State Bank of Metropolis, Metropolis, IL was closed by the Office of the Comptroller of the Currency, and the Federal Deposit Insurance Corporation (FDIC) was named Receiver. As Receiver, the FDIC is charged with winding up the business affairs of the failed financial institution. This includes the disposition of assets and liabilities of the failed financial institution and payment of dividends to approved creditors in order of priority. The FDIC, as Receiver, has taken all necessary actions to conclude the affairs of the failed financial institution, made all dividend distributions as required by law and the receivership estate is deemed terminated. Back to top The FDIC issued a press release about the institution’s closure.  If you represent a media outlet and would like information about the closure, please contact the FDIC Public Affairs Office at (202) 898-6993